In [11]:
import pandas as pd 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf 
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping
import pickle


In [3]:
data = pd.read_csv('Churn_Modelling.csv')
data

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [4]:
data.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='str')

In [5]:
data.drop(columns={'RowNumber', 'CustomerId', 'Surname'}, inplace=True)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [6]:
le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender'] )

one_hot_encoder = OneHotEncoder(handle_unknown='ignore')
geo_OHE = one_hot_encoder.fit_transform(data[['Geography']]).toarray()
geo_OHE_df = pd.DataFrame(geo_OHE, columns=one_hot_encoder.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_OHE_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [10]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [12]:
with open('LE.pkl', 'wb') as file :
    pickle.dump(le, file)

with open('OHE.pkl', 'wb') as file :
    pickle.dump(one_hot_encoder, file)

with open('SC.pkl', 'wb') as file :
    pickle.dump(sc, file)


In [17]:
# Define a function to create a model and try differnce parameter

def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))

    for _ in range(layers-1) :
        model.add(Dense(neurons, activation='relu'))
    
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [36]:
# create a keras classifier

model = KerasClassifier(model=create_model, model__layers=1, model__neurons=32, epochs=50, batch_size=10, verbose=0)

In [37]:
# define the grid search parameter -
param_grid = {
    'model__neurons' : [16, 32, 64, 128],
    'model__layers' : [1, 2],
    'epochs' : [50, 10]
}

In [38]:
# Perform grid search

grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3, verbose=0)
grid_result = grid.fit(X_train, y_train)

# printing the best parameter
print("Best: %f using %s" %(grid_result.best_score_, grid_result.best_params_))

c:\Users\nagka\Desktop\ANN CLASSIFICATION\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best: 0.860124 using {'epochs': 10, 'model__layers': 2, 'model__neurons': 128}
